# Natural Language Processing - Assignment 1
## Track A: Short Answer Questions (SAQ)
### Cross-Cultural Knowledge Evaluation

**Student Name**: (John) Paul Nagle  
**Student ID**: R00065426  
**Model**: Mistral-7B-Instruct-v0.2  
**Locales**: ga-IE (Irish), en-US (English-US), ar-SA (Arabic-Saudi Arabia), zh-CN (Chinese-China)

## Setup and Installation

In [110]:
# Install required packages
!pip install transformers datasets accelerate torch sentencepiece bitsandbytes -q


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## Imports and Configuration

In [111]:
import warnings
import re
import unicodedata
import json
import random
import numpy as np
from typing import Dict, List, Optional, Literal

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

warnings.filterwarnings("ignore")

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.11.0
CUDA available: False


## Step 1: Locale Configuration

Selected locales meet assignment requirements:
- **en-US** (English-US): High-resource baseline
- **en-GB** (English-UK): High-resource, European locale
- **zh-CN** (Chinese-China): Non-Latin script, major language
- **am-ET** (Amharic-Ethiopia): Low-resource, under-represented locale

In [112]:
# Define locales
LOCALES = {
    'en-US': {
        'code': 'en-US',
        'name': 'English (United States)',
        'language': 'English',
        'script': 'Latin',
        'resource_level': 'high'
    },
    'en-GB': {
        'code': 'en-GB',
        'name': 'English (United Kingdom)',
        'language': 'English',
        'script': 'Latin',
        'resource_level': 'high'
    },
    'zh-CN': {
        'code': 'zh-CN',
        'name': 'Chinese (China)',
        'language': 'Simplified Chinese',
        'script': 'Han',
        'resource_level': 'high'
    },
    'am-ET': {
        'code': 'am-ET',
        'name': 'Amharic (Ethiopia)',
        'language': 'Amharic',
        'script': 'Ethiopic',
        'resource_level': 'low'
    }
}

print("Configured Locales:")
for locale_code, config in LOCALES.items():
    print(f"  {locale_code}: {config['name']} ({config['script']} script, {config['resource_level']}-resource)")

Configured Locales:
  en-US: English (United States) (Latin script, high-resource)
  en-GB: English (United Kingdom) (Latin script, high-resource)
  zh-CN: Chinese (China) (Han script, high-resource)
  am-ET: Amharic (Ethiopia) (Ethiopic script, low-resource)


## Step 2: Load Mistral-7B Model

**Hardware Detection & Model Loading Strategy:**
- GPU available: Use 4-bit quantization with Mistral-7B
- CPU only: Use smaller model (TinyLlama-1.1B) for faster inference

**Note**: For CPU-only systems, TinyLlama is recommended. For production with GPU, use Mistral-7B.

In [113]:
# Detect hardware and choose appropriate configuration
USE_GPU = torch.cuda.is_available()
USE_SMALLER_MODEL = not USE_GPU  # Use smaller model on CPU for speed

# Model selection based on hardware
if USE_SMALLER_MODEL:
    MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # 1.1B params, faster on CPU
    print("⚠️  CPU detected: Using TinyLlama-1.1B for faster inference")
    print("   (For production, use Mistral-7B on GPU)")
else:
    MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"  # 7B params
    print("✓ GPU detected: Using Mistral-7B-Instruct-v0.3")

TEMPERATURE = 0.0  # Required for reproducibility
MAX_NEW_TOKENS = 100

print(f"\nLoading model: {MODEL_NAME}")
print("This may take several minutes...\n")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

# Configure model loading based on hardware
if USE_GPU:
    # GPU: Use 4-bit quantization
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )
    
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=quantization_config,
        device_map="auto",
        trust_remote_code=True
    )
    print("✓ Model loaded with 4-bit quantization on GPU")
    
else:
    # CPU: Load smaller model without quantization
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float32,  # Use float32 for CPU
        device_map="cpu",
        trust_remote_code=True,
        low_cpu_mem_usage=True
    )
    print("✓ Model loaded on CPU (float32)")

model.eval()
print(f"✓ Model: {MODEL_NAME}")
print(f"✓ Device: {'GPU' if USE_GPU else 'CPU'}")
print(f"✓ Temperature: {TEMPERATURE} (deterministic)")
print(f"✓ Max new tokens: {MAX_NEW_TOKENS}")

⚠️  CPU detected: Using TinyLlama-1.1B for faster inference
   (For production, use Mistral-7B on GPU)

Loading model: TinyLlama/TinyLlama-1.1B-Chat-v1.0
This may take several minutes...



Loading weights: 100%|██████████| 201/201 [00:03<00:00, 58.47it/s]


✓ Model loaded on CPU (float32)
✓ Model: TinyLlama/TinyLlama-1.1B-Chat-v1.0
✓ Device: CPU
✓ Temperature: 0.0 (deterministic)
✓ Max new tokens: 100


## Step 3: Text Normalization

Robust normalization strategy for multilingual text matching.

In [114]:
class TextNormalizer:
    """Multi-stage text normalization for answer matching."""

    def __init__(self):
        self.normalization_form: Literal['NFC', 'NFD', 'NFKC', 'NFKD'] = 'NFC'  # Unicode normalization form


    def normalize(self, text: str, locale: Optional[str] = None) -> str:
        """Normalize text for multilingual text matching."""
        if not text:
            return ""

        # 1. Unicode normalization (NFC - Canonical Composition)
        text = unicodedata.normalize(self.normalization_form, text)

        # 2. Remove control characters
        text = re.sub(r'[\x00-\x08\x0B-\x0C\x0E-\x1F\x7F-\x9F]', '', text)

        # 3. Normalize line breaks
        text = text.replace('\r\n', '\n').replace('\r', '\n')

        # 4. Normalize whitespace (but preserve single spaces)
        text = re.sub(r'[ \t]+', ' ', text)
        text = re.sub(r'\n{3,}', '\n\n', text)

        # 5. Strip leading/trailing whitespace
        text = text.strip()

        # 6. Convert all to lower case
        text = text.lower()

        # 7. Remove punctuation (but keep apostrophes for contractions)
        text = re.sub(r'[^\w\s\'\-]', '', text)

        # 8. Normalize multiple spaces
        text = re.sub(r'\s+', ' ', text)

        return text


# Initialize normalizer
normalizer = TextNormalizer()


## Step 4: Baseline SAQ System

Direct prompting baseline with locale-aware generation.

In [115]:
class BaselineSAQSystem:
    """Baseline Short Answer Question system using direct prompting."""
    def __init__(self, model, tokenizer, normalizer, temperature=0.0):
        self.model = model
        self.tokenizer = tokenizer
        self.normalizer = normalizer
        self.temperature = temperature
        self.max_new_tokens = 100
    
    def create_prompt(self, question: str, locale: str) -> str:
        """ Create a prompt for the model. """
        locale_config = LOCALES.get(locale)
        if not locale_config:
            raise ValueError(f"Unknown locale: {locale}")
        
        # TinyLlama chat format
        if "TinyLlama" in MODEL_NAME:
            prompt = f"""<|system|>
    You are a helpful assistant that answers questions about {locale_config['name']} culture and everyday knowledge.</s>
    <|user|>
    {question}</s>
    <|assistant|>
    """
        else:
            # Mistral/Llama format
            prompt = f"""[INST] Answer the following question about {locale_config['name']} culture and everyday knowledge.
    Provide a short, direct answer in {locale_config['language']}.

    Question: {question}

    Answer: [/INST]"""
    
        return prompt
    
    def generate_answer(self, question: str, locale: str) -> Dict:
        """ Generate answer for a question in the specified locale. """
        # Normalize question
        question = self.normalizer.normalize(question)
        
        # Create prompt
        prompt = self.create_prompt(question, locale)
        
        # Tokenize
        inputs = self.tokenizer(
            prompt,
            return_tensors="pt",
            padding=True,
            truncation=True
        ).to(self.model.device)
        
        # Generate with temperature=0 for reproducibility
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                max_length=None, 
                temperature=self.temperature if self.temperature > 0 else None,
                do_sample=False,  # Greedy decoding when temperature=0
                pad_token_id=self.tokenizer.eos_token_id,
                eos_token_id=self.tokenizer.eos_token_id
            )

        # Decode only the new tokens (not the prompt)
        generated_tokens = outputs[0][inputs['input_ids'].shape[1]:]
        answer = self.tokenizer.decode(generated_tokens, skip_special_tokens=True)
        
        # Normalize answer
        answer = self.normalizer.normalize(answer)
        
        return {
            'question': question,
            'locale': locale,
            'answer': answer,
            'raw_output': answer,
            'prompt': prompt
        }

# Initialize baseline system
baseline_system = BaselineSAQSystem(
    model=model,
    tokenizer=tokenizer,
    normalizer=normalizer,
    temperature=TEMPERATURE
)

print("✓ Baseline SAQ System initialized")
print(f"  Model: {MODEL_NAME}")
print(f"  Temperature: {TEMPERATURE}")
print(f"  Locales: {', '.join(LOCALES.keys())}")

✓ Baseline SAQ System initialized
  Model: TinyLlama/TinyLlama-1.1B-Chat-v1.0
  Temperature: 0.0
  Locales: en-US, en-GB, zh-CN, am-ET


## Example Questions for Testing

Culture-specific questions for each locale.

In [116]:
import pandas as pd

# Load questions from CSV files
def load_questions_from_csv(locale_code, num_questions=10):
    """Load first N questions from locale-specific CSV file"""
    locale_to_file = {
        'en-US': 'US_questions.csv',
        'en-GB': 'UK_questions.csv', 
        'zh-CN': 'China_questions.csv',
        'am-ET': 'Ethiopia_questions.csv'
    }
    
    filename = locale_to_file.get(locale_code)
    if not filename:
        return []
    
    try:
        df = pd.read_csv(f'questions/{filename}')
        questions = df['Question'].head(num_questions).tolist()
        return questions
    except Exception as e:
        print(f"Error loading {filename}: {e}")
        return []


## Test Baseline System

Generate answers for sample questions.

In [117]:
# Load first 10 questions for each locale
SAMPLE_QUESTIONS = {
    locale_code: load_questions_from_csv(locale_code, 10)
    for locale_code in ['en-US', 'en-GB', 'zh-CN', 'am-ET']
}

# Test with 10 questions for each locale
print("Testing Baseline System:\n")
print("=" * 80)

for locale in LOCALES.keys():
    print(f"\n{'='*80}")
    print(f"Locale: {locale} ({LOCALES[locale]['name']})")
    print(f"{'='*80}\n")
    
    for i, question in enumerate(SAMPLE_QUESTIONS[locale], 1):
        print(f"{i}.")
        print(f"[Question ]: {question}")
        
        result = baseline_system.generate_answer(question, locale)
        
        print(f"[Answer   ]: {result['answer']}")
        print("-" * 80)

Testing Baseline System:


Locale: en-US (English (United States))

1.
[Question ]: What is a common snack for preschool kids in the US?
[Answer   ]: 1 cheerios 2 rice cakes with peanut butter 3 apple slices with almond butter 4 banana slices with peanut butter 5 yogurt with granola and fruit 6 trail mix with nuts and dried fruit 7 popcorn 8 hard-boiled eggs 9 rice cakes with avocado and
--------------------------------------------------------------------------------
2.
[Question ]: What is a popular food to go with beer in the US?
[Answer   ]: 1 pizza 2 nachos 3 fries 4 mac and cheese 5 chicken wings 6 grilled cheese sandwich 7 tacos 8 burritos 9 poutine canadian fries and gravy 10 hot dogs 11 sushi rolls 12 fried chicken 1
--------------------------------------------------------------------------------
3.
[Question ]: What is the most popular fruit in the US?
[Answer   ]: 1 apple 2 banana 3 oranges 4 peaches 5 pineapples 6 mangoes 7 kiwi 8 strawberries 9 blueberries 10 grapes 11 wate

## Save Configuration

Document all settings for reproducibility.

In [118]:
config = {
    'model_name': MODEL_NAME,
    'temperature': TEMPERATURE,
    'max_new_tokens': MAX_NEW_TOKENS,
    'seed': SEED,
    'locales': LOCALES,
    'normalization_form': normalizer.normalization_form,
    'pytorch_version': torch.__version__,
    'cuda_available': torch.cuda.is_available()
}

# Save configuration
with open('baseline_config.json', 'w', encoding='utf-8') as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

print("✓ Configuration saved to baseline_config.json")
print("\nConfiguration Summary:")
print(json.dumps(config, indent=2, ensure_ascii=False))

✓ Configuration saved to baseline_config.json

Configuration Summary:
{
  "model_name": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
  "temperature": 0.0,
  "max_new_tokens": 100,
  "seed": 42,
  "locales": {
    "en-US": {
      "code": "en-US",
      "name": "English (United States)",
      "language": "English",
      "script": "Latin",
      "resource_level": "high"
    },
    "en-GB": {
      "code": "en-GB",
      "name": "English (United Kingdom)",
      "language": "English",
      "script": "Latin",
      "resource_level": "high"
    },
    "zh-CN": {
      "code": "zh-CN",
      "name": "Chinese (China)",
      "language": "Simplified Chinese",
      "script": "Han",
      "resource_level": "high"
    },
    "am-ET": {
      "code": "am-ET",
      "name": "Amharic (Ethiopia)",
      "language": "Amharic",
      "script": "Ethiopic",
      "resource_level": "low"
    }
  },
  "normalization_form": "NFC",
  "pytorch_version": "2.11.0",
  "cuda_available": false
}


## Next Steps

**Completed (Steps 1-4):**
1. ✓ Locale selection (en-US, en-GB, zh-CN, am-ET)
2. ✓ Mistral-7B model loaded with 4-bit quantization
3. ✓ Text normalization pipeline implemented
4. ✓ Baseline SAQ system with direct prompting

**Remaining (Steps 5-8):**
5. Implement 2+ improvements (e.g., locale-aware prompting, confidence estimation)
6. Evaluate across locales with performance metrics
7. Analyze 10+ failure examples
8. Write 6-10 page report with Responsible AI section